In [7]:
!pip install flask pyngrok -q

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import pickle
import numpy as np
import threading

app = Flask(__name__)

with open('/content/drive/MyDrive/aethercept_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('/content/drive/MyDrive/aethercept_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

FEATURES = [
    'click_count', 'avg_click_interval', 'click_interval_variance',
    'click_interval_entropy', 'mouse_velocity_variance',
    'max_element_click_rate', 'scroll_events', 'keystroke_count'
]

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    features = [[data[f] for f in FEATURES]]
    scaled = scaler.transform(features)
    pred = model.predict(scaled)[0]
    score = model.decision_function(scaled)[0]
    return jsonify({
        'prediction': 'bot' if pred == -1 else 'human',
        'anomaly_score': round(float(score), 4)
    })

@app.route('/')
def home():
    return jsonify({'status': 'AetherCept API running ✅'})

ngrok.set_auth_token('3BtyfcQxKIU3cO29jyXhfzHf3LI_5W4DzEvi4eg1eXEJ6EKpg')
public_url = ngrok.connect(5000)
print(f"Public URL: {public_url}")


threading.Thread(target=app.run, kwargs={'port': 5000}).start()

Public URL: NgrokTunnel: "https://intersubjective-alyce-illusively.ngrok-free.dev" -> "http://localhost:5000"


#validation

In [14]:
import requests

url = str(public_url).replace("NgrokTunnel: \"", "").replace("\" -> \"http://localhost:5000\"", "")

response = requests.post(f"{url}/predict", json={
    'click_count': 5,
    'avg_click_interval': 1200,
    'click_interval_variance': 200000,
    'click_interval_entropy': 3.0,
    'mouse_velocity_variance': 0.55,
    'max_element_click_rate': 1.0,
    'scroll_events': 10,
    'keystroke_count': 8
})

print(response.json())

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:18:20] "POST /predict HTTP/1.1" 200 -


{'anomaly_score': 0.1471, 'prediction': 'human'}
